# Focused hydrograph case review
Review **one case at a time**, ordered by error or observed shift. Start with 37's delays, then 22's unchanged cases, and 55's full compounds. Networks 15 and 37 also contain many unchanged cases.

The buffer is a review filter, not a fitted classifier. Positive observed shift means later child maximum; signed error is **predicted − observed**. “Delay outside buffer” means larger than the chosen buffer, not statistical significance. Native hydrographs retain the global-maximum definition. Example 54 can switch crest dominance.

These are saved held-out predictions; no models are refitted. Rank curvature alone does not establish a nonlinear physical relationship. The optional scatter uses actual predictor values.

In [ ]:
from pathlib import Path
import sys, importlib
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'RAPID/src').exists())
sys.path.insert(0, str(ROOT/'RAPID/src'))
import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as w
from IPython.display import display, clear_output
import rapid_tools.current_domains as domains
import rapid_tools.current_review as review
importlib.reload(domains); importlib.reload(review)
rows = pd.read_parquet(ROOT/'RAPID/outputs/hydrograph_residual_review_20261018/diagnostic_rows.parquet')
rows['example'] = rows.example.astype(str).str.zfill(2)
cache = {}


In [ ]:
scenario = w.Dropdown(options=sorted(rows.scenario.unique()), value='all', description='Scenario')
network = w.Dropdown(options=sorted(rows.example.unique()), value='37', description='Network')
kind = w.Dropdown(options=['All', *sorted(rows.edit_type.unique())], description='Type')
direction = w.Dropdown(options=['All', 'Exactly unchanged', 'Within buffer', 'Delay outside buffer', 'Advance outside buffer'], value='Delay outside buffer', description='Shift')
buffer = w.FloatText(value=1., description='Buffer (min)')
order = w.Dropdown(options=['Largest absolute error', 'Largest delay', 'Largest advance'], description='Order')
case = w.Dropdown(description='Case', layout=w.Layout(width='95%'))
show_map = w.Checkbox(value=True, description='Show domain map')
show_scatter = w.Checkbox(value=False, description='Show filtered length/error scatter')
button = w.Button(description='Inspect selected case', button_style='primary')
count = w.HTML(); output = w.Output()

def filtered():
    d = rows[rows.scenario.eq(scenario.value) & rows.example.eq(network.value)].copy()
    if kind.value != 'All': d = d[d.edit_type.eq(kind.value)]
    minutes = d.observed_hours * 60
    threshold = max(0., buffer.value)
    mask = {'All': pd.Series(True,index=d.index),
            'Exactly unchanged': minutes.eq(0), 'Within buffer': minutes.abs().le(threshold),
            'Delay outside buffer': minutes.gt(threshold),
            'Advance outside buffer': minutes.lt(-threshold)}[direction.value]
    d = d[mask]
    if order.value == 'Largest absolute error': return d.sort_values('abs_error_min', ascending=False)
    return d.sort_values('observed_hours', ascending=order.value == 'Largest advance')

def refresh(*_):
    d = filtered()
    case.options = [(f'{r.state} | {r.edit_type} | observed {60*r.observed_hours:+.1f} min | error {r.error_min:+.1f} min', i) for i,r in d.iterrows()]
    case.value = int(d.index[0]) if len(d) else None
    count.value = f'<b>{len(d)} cases</b>. Select a case and click Inspect; only that case is rendered.'

def inspect(_):
    with output:
        clear_output(wait=True)
        if case.value is None:
            print('No cases match these filters.'); return
        r = rows.loc[case.value]
        if r.example not in cache: cache[r.example] = review.build_current_analysis_frame([r.example])
        active = cache[r.example]
        aliases = {r.state, *str(r.aliases).split(' | ')}
        matches = active[active.state_id.isin(aliases)]
        if matches.empty: raise ValueError(f'No current output matches {r.example}/{r.state}; aliases: {aliases}')
        exact = matches[matches.state_id.eq(r.state)]
        selected = (exact if len(exact) else matches).iloc[0]
        if not np.isclose(selected.sampled_peak_shift_minutes, 60*r.observed_hours, atol=1e-6):
            raise ValueError('Current peak shift differs from saved diagnostic response; refresh diagnostics before comparison.')
        display(pd.Series({'canonical state':r.state, 'plotted state':selected.state_id,
                           'observed shift (min)':60*r.observed_hours,
                           'predicted shift (min)':60*r.predicted_hours,
                           'signed error (min)':r.error_min}).to_frame('Value'))
        review.parent_child_hydrograph_figure(selected, show_forcing=True).show()
        if show_map.value:
            try: review.parent_child_network_figure(selected, zoom_to_domain=True).show()
            except (ValueError, FileNotFoundError) as exc: print(f'Map unavailable: {exc}')
        if show_scatter.value:
            d = filtered()
            px.scatter(d, x='B_parent_total_length', y='error_min', color='edit_type',
                       hover_data=['state','observed_hours','predicted_hours'],
                       labels={'B_parent_total_length':'Parent outer-domain total length (m)',
                               'error_min':'Predicted − observed (min)'}, template='plotly_white').show()

for control in [scenario, network, kind, direction, buffer, order]: control.observe(refresh, names='value')
button.on_click(inspect)
refresh()
display(w.VBox([w.HBox([scenario, network, kind]), w.HBox([direction, buffer, order]),
                count, case, w.HBox([show_map, show_scatter]), button, output]))
